# Pose completion: our models against the others, on their Pareto fronts

The completions of PoseD-Flow's protocol (four occlusions, K = 10 hypotheses per pose), one split at a time (a
selector above the plots):

- **validation**: every setting tried so far on the fixed validation poses, by the validation scripts of
  `explore/evaluation/completion_diagnostics` and the Optuna studies of `scripts/tune_completion.py`; settings are
  chosen here;
- **test**: the rows of the completion cache on the 5000 test poses (`scripts/evaluate_completion.py` and the test
  scripts of the diagnostics), each a setting chosen on validation and run once; only to report, never to choose.

Both are read in one form by `hp.evaluation.completion_results` (`load`, `load_test`); the two splits are never drawn
together.

**The question: is RSGM (ours) as good as DPoser and PoseD-Flow?** Each family of models has a colour; its front is
drawn through the best of all its settings, whatever the network and the completion method (the marker shows the
method: inpainting, D-Flow, DPoser's optimization). One family is better than another where its front lies on the
better side (left and up).

A single number does not rank the settings: the hypotheses trade accuracy for diversity. The mean MPVPE of K
hypotheses is lowest when they collapse onto one pose, the best of K and the APD (the mean distance between the
hypotheses of a pose) grow with their spread. Hence two plots, both against the diversity:

- **mean ↓ against APD ↑**: accuracy against diversity. APD alone is not a goal (random poses have the largest APD);
  the dotted line is the APD of D-Flow (Heun EDM 50, RSGM small std c10) on the 1000 poses, the floor of the test rule
  of the inpainting studies.
- **best of 10 ↓ against APD ↑**: whether the spread covers the plausible completions.

Only results on the same poses compare: the first 100 validation poses (the smoke tests on the Mac) are not the 1000 of
the studies, and neither are the 5000 test poses.

In [ ]:
import html
import io
import sys
from pathlib import Path

# the code of the checkout this notebook is in (e.g. a worktree), before an editable install of another one
SRC = next(str(p / "src") for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hp").is_dir())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from hp.evaluation import completion_results as R
from hp.evaluation import fronts as F
from hp.evaluation.completion import OCCLUSIONS
from hp.visualization import fronts as V

assert sys.modules["hp.evaluation.fronts"].__file__.startswith(SRC), \
    f"hp comes from {sys.modules['hp.evaluation.fronts'].__file__}: restart the kernel"

# ---- parameters ----------------------------------------------------------------------
RESULTS = Path("~/data/experiments/completion").expanduser()  # the .jsonl of the scripts, optuna/*.db of the studies
SOURCES = sorted(RESULTS.glob("*.jsonl")) + sorted((RESULTS / "optuna").glob("*.db"))  # copies from the servers too
CACHE = RESULTS / "cache"  # the test rows (cache/<label>/<occlusion>.pt and _metrics.pt)
SPLIT = "validation"  # the split shown at first: "validation" or "test"
POSES = 1000  # the validation poses shown at first (a selector offers every number in the results)

E = "~/data/experiments/"
NAMES = {  # the run folders of the studies: the names of dflow_valid.MODELS
    E + "rsgm_posedflow_small_s10/run_0001": "small std c10",
    E + "rsgm_posedflow_deep_s10/run_0001": "deep",
    E + "rsgm_posedflow_abl_residual_s10/run_0001": "abl residual",
    E + "rsgm_posedflow_abl_norm_s10/run_0001": "abl norm",
    E + "rsgm_posedflow_res_norm_s10/run_0001": "res norm",
    E + "full/rsgm_posedflow_dposer_like_s10/run_0001": "dposer like full",
    E + "full/rsgm_posedflow_dposer_like_z_s10/run_0001": "dposer like z full",
    E + "full/rsgm_posedflow_small_s10/run_0001": "small std full",
    E + "full/rsgm_posedflow_small_lik_s10/run_0001": "small lik full",
    E + "full/rsgm_posedflow_search_lik_s10/run_0001": "search lik full",
    E + "dposer_posedflow_rsgm_budget_s10/run_0001": "DPoser c10",
    E + "full/dposer_posedflow_s10/run_0001": "DPoser full",
    E + "full/rsgm_posedflow_dposer_like_aniso_s10/run_0001": "dposer like aniso full",
    E + "igso_speed/rsgm_posedflow_abl_norm_aniso_s10/run_0001": "abl norm aniso",
    "~/data/posedflow_ckpt/checkpoints/epoch-009_step-0_loss-0.0000.ckpt": "PoseD-Flow: their checkpoint",
}
# the families of models and their colours (those of pareto_posedflow_vs_rsgm.ipynb); a model is of the first family
# whose prefix its name starts with, else RSGM
FAMILIES = {"RSGM (ours)": "#e34948", "DPoser": "#7b3fa0", "PoseD-Flow": "#2a78d6"}
PREFIXES = {"DPoser": "DPoser", "PoseD-Flow": "PoseD-Flow", "posedflow": "PoseD-Flow"}
METRICS = {  # key of a record: (axis label, direction)
    "mean": ("MPVPE mean (mm)", "min"),
    "best_of_k": ("MPVPE best of 10 (mm)", "min"),
    "apd": ("APD (cm)", "max"),
}
PANELS = [("mean", "apd"), ("best_of_k", "apd")]  # (x, y) of each plot
APD_FLOOR = 6.40  # APD of D-Flow (Heun EDM 50, small std c10) on the 1000 validation poses: the test rule's floor
#                  (drawn on validation only)
MARKERS = {"inpaint": "o", "dflow": "*", "dposer": "s"}  # per method
SHORT = {"inpaint": "inpaint", "dflow": "D-Flow", "dposer": "DPoser opt."}  # per method, in the labels of the points
LIMITS = {"mean": 140, "best_of_k": 80}  # the view ends here (mm): settings beyond are counted, not drawn (the fronts
#                                          are of all of them; the degenerate ones, near random poses, go far beyond)
GROUPS = "family x method"  # at first a front per "family" (whatever the method) or per "family x method"
DASHES = {"inpaint": "-", "dflow": "--", "dposer": ":"}  # per method, the lines of the fronts per family x method
FRONT_STYLE = "steps"  # "steps", "lines" or "points" (V.STYLES)
MODEL_FRONTS = False  # at first also a thin front for every model, besides the front of each family
LABELS = False  # at first labels on the points of the family fronts
FIGURES = RESULTS / "figures"  # the default folder of the save button
# --------------------------------------------------------------------------------------

## 1. The results

Every record is one setting of one completion method on one model. The same setting may be in a script and in a
study: both are kept, they are separate runs.

In [ ]:
def family_of(model):
    return next((f for prefix, f in PREFIXES.items() if model.startswith(prefix)), "RSGM (ours)")


records = [{**r, "split": "validation"} for r in R.load(SOURCES)] + R.load_test(CACHE)
for r in records:
    r["model"] = NAMES.get(r["model"], r["model"])
    r["family"] = family_of(r["model"])
MODELS = sorted({r["model"] for r in records}, key=lambda m: (list(FAMILIES).index(family_of(m)), m))

counts = {}
for r in records:
    counts.setdefault((r["split"], r["poses"], r["family"], r["model"], r["method"]), []).append(r)
print(f"{len(records)} records from {len(SOURCES)} files and the test cache {CACHE}")
for (split, poses, family, model, method), rows in sorted(
        counts.items(), key=lambda kv: (kv[0][0] == "test", kv[0][1], list(FAMILIES).index(kv[0][2]), kv[0][3:])):
    sources = ", ".join(sorted({r["source"].split(":")[0] for r in rows}))
    print(f"  {split:10s} {poses:5d} poses  {family:12s} {model:40s} {R.METHODS[method].split(' (')[0]:22s} "
          f"{len(rows):4d}  ({sources})")

## 1a. What the studies varied

Every Optuna study (`config/completion_tuning/<study>.yaml`, `scripts/tune_completion.py`) fixes a model and a method
and searches the settings of the method on the same 1000 validation poses with the same seeds; the table below is read
from the configurations the studies recorded. The knobs of each method, and which way they move a setting along the
front (from accurate and alike hypotheses to diverse ones):

- **inpainting** (`hp.models.inpainting`; RSGM and DPoser): `steps` of the reverse process, `guidance` ζ (the
  gradient of the error on the visible joints, DPS-like: larger is closer to the observation, less diverse),
  `resample` r (RePaint: r passes per step, more consistent hidden joints), `replace` (the visible joints replaced by
  the noised observation at every step), `schedule` (fixed, EDM).
- **D-Flow** (`hp.models.dflow`; PoseD-Flow, RSGM): `lr` and `steps` of the optimizer of the start of the ODE
  (fewer or smaller steps: closer to samples of the prior, more diverse), `running_weight` (their running cost, keeps
  the trajectory near plausible rotations); fixed: the ODE (PoseD-Flow their Euler with 100 steps, RSGM Heun EDM with
  25 steps, 50 network calls), `invert` (off).
- **DPoser's method** (`hp.models.score_completion`; DPoser, and on SO(3) for RSGM as score distillation): `lr`,
  the window of noise levels `t_max`, `t_min = t_max * t_min_fraction` (wider and higher: more diverse),
  `iterations` × `steps_per_iter` (DPoser), `prior_weight` (RSGM), the start (`noise`) and `scale` (RSGM, fixed).

The other records (the `.jsonl` of the validation scripts) are single settings, not searches; the studies made before
the configuration was recorded (`completion_<method>_<run>_v<poses>`) are not listed.

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def searched(spec):
    if isinstance(spec, list):
        return "{" + ", ".join(map(str, spec)) + "}"
    return f"[{spec['low']:g}, {spec['high']:g}]" + (" log" if spec.get("log") else "")


print(f"{'study':46s} {'model':28s} {'method':12s} {'objective':9s} {'trials':>6s}  searched; fixed")
for path in (p for p in SOURCES if p.suffix == ".db"):
    for summary in optuna.get_all_study_summaries(f"sqlite:///{path}"):
        study = optuna.load_study(study_name=summary.study_name, storage=f"sqlite:///{path}")
        config = study.user_attrs.get("config")
        if not config:  # older studies, before the configuration was recorded
            continue
        done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
        knobs = ", ".join(f"{k} {searched(v)}" for k, v in config["params"].items())
        fixed = ", ".join(f"{k}={v}" for k, v in (config.get("fixed") or {}).items())
        print(f"{summary.study_name[:46]:46s} {NAMES.get(config['model'], config['model'])[:28]:28s} "
              f"{SHORT[config['method']]:12s} {config['objective']:9s} {done:6d}  {knobs}" + (f"; {fixed}" if fixed else ""))

## 2. The fronts of each family, in numbers

The settings on the front of each family (mean against APD, the first plot), from the most accurate to the most
diverse, with the model, the method and all three metrics. The selectors of section 3 change the poses, the occlusion
and the models of this table too.

In [ ]:
def at(rows, occlusion):
    """The records with the metrics of one occlusion (None: the means over the occlusions)."""
    return [R.at_occlusion(r, occlusion) for r in rows]


def short(options):
    return ", ".join(f"{k}={v:.3g}" if isinstance(v, float) else f"{k}={v}" for k, v in sorted(options.items()))


def objectives(x, y):
    return (x, METRICS[x][1]), (y, METRICS[y][1])


def groups_of(rows, by):
    """The groups of records with a front each, in the order of FAMILIES then MARKERS: {name: (family, method or None,
    records)}; by "family" or "family x method"."""
    out = {}
    for family in FAMILIES:
        for method in ([None] if by == "family" else list(MARKERS)):
            mine = [r for r in rows if r["family"] == family and method in (None, r["method"])]
            if mine:
                out[family if method is None else f"{family} + {SHORT[method]}"] = (family, method, mine)
    return out


def table(rows, x="mean", y="apd", by="family"):
    out = io.StringIO()
    for name, (family, method, mine) in groups_of(rows, by).items():
        print(f"\n{name}: {len(mine)} settings of {len({r['model'] for r in mine})} models; front of {x} and {y}:",
              file=out)
        print(f"  {'mean':>7s} {'best10':>7s} {'APD':>6s}  {'model':30s} {'method':12s} setting", file=out)
        for r in F.front(mine, *objectives(x, y)):
            print(f"  {r['mean']:7.2f} {r['best_of_k']:7.2f} {r['apd']:6.2f}  {r['model'][:30]:30s} "
                  f"{SHORT[r['method']]:12s} {r['setting']} ({r['source'].split(':')[-1]}) {short(r['options'])}",
                  file=out)
    return out.getvalue()


print(table(at([r for r in records if r["split"] == SPLIT and r["poses"] == POSES], None), by=GROUPS))

## 3. The plots

Each family in its colour: the points of all its settings (faint; the marker is the method) and its front (solid,
through its best settings). Optional: a thin dashed front for every model, and labels on the points of the family
fronts. Above the plots: the split (validation or test) and its poses, the occlusion (or the mean over the four), the models drawn, the style
of the fronts, and the save button (the format by the extension: .png, .pdf, .svg). A click on a point (with
ipyevents, e.g. JupyterLab) highlights its model.

In [ ]:
def figure(rows, models, occlusion, style, model_fronts, labelled, split="validation", by="family"):
    """The plots of PANELS for the records `rows` (of one split) of the models `models`, a front per group of `by`
    ("family", or "family x method": the colour of the family, the line of the method)."""
    rows = [r for r in at(rows, occlusion) if r["model"] in models]
    families = [f for f in FAMILIES if any(r["family"] == f for r in rows)]
    groups = groups_of(rows, by)
    drawn = []
    with plt.style.context("default"):
        fig, axes = plt.subplots(1, len(PANELS), figsize=(7 * len(PANELS), 6))
        axes = np.atleast_1d(axes)
        labels = {ax: [] for ax in axes}
        for ax, (x, y) in zip(axes, PANELS):
            seen = [r for r in rows if all(r[m] <= LIMITS.get(m, np.inf) for m in (x, y))]
            for family in families:
                color = FAMILIES[family]
                for r in (r for r in rows if r["family"] == family and r in seen):
                    artist, = ax.plot(r[x], r[y], marker=MARKERS[r["method"]], ls="", ms=6 + 3 * (r["method"] == "dflow"),
                                      alpha=0.35, color=color, markeredgecolor="white", markeredgewidth=0.5, zorder=2)
                    drawn.append((artist, r[x], r[y], r["model"]))
            for name, (family, method, mine) in groups.items():
                color, dash = FAMILIES[family], DASHES.get(method, "-")
                if model_fronts:
                    for model in sorted({r["model"] for r in mine}):
                        front = F.front([r for r in mine if r["model"] == model], *objectives(x, y))
                        if len(front) > 1:
                            V.draw_front(ax, [r[x] for r in front], [r[y] for r in front], style, color=color,
                                         ls="--", lw=0.8, ring=10, corner="x")
                front = F.front(mine, *objectives(x, y))
                V.draw_front(ax, [r[x] for r in front], [r[y] for r in front], style, color=color, ls=dash, lw=2.2,
                             ring=14, corner="x")
                for r in front:  # the points of the front: opaque
                    if r in seen:
                        ax.plot(r[x], r[y], marker=MARKERS[r["method"]], ls="", ms=8 + 4 * (r["method"] == "dflow"),
                                color=color, markeredgecolor="white", markeredgewidth=0.8, zorder=3)
                if labelled:
                    labels[ax] += [(r[x], r[y], f"{r['model']}, {SHORT[r['method']]}: {r['setting']}")
                                   for r in front if r in seen]
            if occlusion is None and split == "validation":
                ax.axhline(APD_FLOOR, color="0.5", lw=1, ls=":", zorder=0)
            if seen and len(seen) < len(rows):
                for set_lim, values in ((ax.set_xlim, [r[x] for r in seen]), (ax.set_ylim, [r[y] for r in seen])):
                    low, high = min(values), max(values)
                    set_lim(low - 0.04 * (high - low), high + 0.04 * (high - low))
                beyond = ", ".join(f"{METRICS[m][0].split(' (')[0]} {LIMITS[m]} mm" for m in (x, y) if m in LIMITS)
                ax.set_title(f"{len(rows) - len(seen)} settings beyond {beyond}: not drawn", fontsize=8.5,
                             color="0.35")
            ax.set_xlabel(f"← better   {METRICS[x][0]}")
            ax.set_ylabel(f"{METRICS[y][0]}   better →")
            ax.grid(True, color="0.92", lw=0.8)
            for side in ("top", "right"):
                ax.spines[side].set_visible(False)
        handles = [plt.Line2D([], [], color=FAMILIES[family], lw=2.2, ls=DASHES.get(method, "-"), label=f"{name}: its front")
                   for name, (family, method, _) in groups.items()]
        handles += [plt.Line2D([], [], marker=m, ls="", color="0.4", ms=7 + 3 * (k == "dflow"),
                               label=R.METHODS[k].split(" (")[0]) for k, m in MARKERS.items()
                    if any(r["method"] == k for r in rows)]
        if occlusion is None and split == "validation":
            handles += [plt.Line2D([], [], color="0.5", ls=":", label=f"APD of RSGM D-Flow Heun EDM 50 ({APD_FLOOR} cm)")]
        fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 4), frameon=False, fontsize=9.5)
        poses = rows[0]["poses"] if rows else "no"
        where = "mean over the occlusions" if occlusion is None else occlusion
        fig.suptitle(f"Pose completion on {poses} {split} poses, K = 10, {where}", fontsize=12)
        fig.tight_layout(rect=(0, 0.05 + 0.03 * ((len(handles) - 1) // 4), 1, 0.96))
        fig.patch.set_facecolor("white")
        if labelled:
            V.place_labels(fig, labels, fontsize=6.5)
    height = fig.bbox.height
    hits = [(a, p[0], height - p[1], m) for a, x, y, m in drawn for p in [a.axes.transData.transform((x, y))]]
    return fig, hits


# ---- the controls ----
L = lambda w: widgets.Layout(width=w)
split_box = widgets.ToggleButtons(options=["validation", "test"], value=SPLIT, description="split",
                                  layout=L("330px"), style={"button_width": "100px"})
poses_box = widgets.Dropdown(options=sorted({r["poses"] for r in records if r["split"] == SPLIT}),
                             value=POSES if SPLIT == "validation" else None, description="poses", layout=L("170px"))
if poses_box.value is None:
    poses_box.value = poses_box.options[-1]
occlusion_box = widgets.Dropdown(options=[("mean of the four", "mean")] + [(o, o) for o in OCCLUSIONS],
                                 description="occlusion", layout=L("260px"))
style_box = widgets.Dropdown(options=[(d, s) for s, d in V.STYLES.items()], value=FRONT_STYLE, description="fronts",
                             layout=L("360px"))
model_fronts_box = widgets.Checkbox(value=MODEL_FRONTS, description="a front per model", indent=False, layout=L("170px"))
labels_box = widgets.Checkbox(value=LABELS, description="labels", indent=False, layout=L("100px"))
groups_box = widgets.ToggleButtons(options=["family", "family x method"], value=GROUPS, description="a front per",
                                   layout=L("420px"), style={"button_width": "130px"})
model_boxes = widgets.HBox()
path_field = widgets.Text(description="save to", layout=L("700px"), style={"description_width": "initial"})
save_button = widgets.Button(description="save", icon="save", layout=L("90px"))
status, picked = widgets.Label(), widgets.Label()
image = widgets.Image(format="png", layout=widgets.Layout(width="100%"))
text = widgets.HTML()
state = {"fig": None, "hits": [], "rings": [], "highlight": None, "default": None}


def current():
    return [r for r in records if r["split"] == split_box.value and r["poses"] == poses_box.value]


def set_split(*_):
    """The poses of the chosen split (the 1000 of the studies if there, else the most), then the models."""
    options = sorted({r["poses"] for r in records if r["split"] == split_box.value})
    value = POSES if POSES in options else options[-1]
    poses_box.unobserve(set_choices, names="value")
    poses_box.options = options
    poses_box.value = value
    poses_box.observe(set_choices, names="value")
    set_choices()


def checked():
    return [box.model for column in model_boxes.children for box in column.children[1:] if box.value]


def set_choices(*_):
    """A column of checkboxes per family: the models of the chosen poses (all checked), then draw."""
    rows = current()
    columns = []
    for family in FAMILIES:
        boxes = []
        for model in MODELS:
            if family_of(model) == family and any(r["model"] == model for r in rows):
                box = widgets.Checkbox(value=True, description=model, indent=False, layout=L("330px"))
                box.model = model
                box.observe(redraw, names="value")
                boxes.append(box)
        if boxes:
            title = widgets.HTML(f"<b style='color: {FAMILIES[family]}'>{family}</b>")
            columns.append(widgets.VBox([title] + boxes))
    model_boxes.children = columns
    redraw()


def redraw(*_):
    try:
        draw()
    except Exception:
        import traceback
        text.value = f"<pre style='color: #b00'>{html.escape(traceback.format_exc())}</pre>"


def draw():
    models = checked()
    occlusion = None if occlusion_box.value == "mean" else occlusion_box.value
    tag = "v" if split_box.value == "validation" else "test"
    default = str(FIGURES / f"completion_fronts_{tag}{poses_box.value}_{occlusion or 'mean'}.png")
    if path_field.value in ("", state["default"]):
        path_field.value = state["default"] = default
    if not models:
        image.value, text.value, state["fig"] = b"", "<pre>no model checked</pre>", None
        return
    fig, hits = figure(current(), models, occlusion, style_box.value, model_fronts_box.value, labels_box.value,
                       split_box.value, groups_box.value)
    plt.close(fig)
    state.update(fig=fig, hits=hits, rings=[])
    show()
    rows = [r for r in at(current(), occlusion) if r["model"] in models]
    text.value = f"<pre style='font-size: 12px; line-height: 1.3'>{html.escape(table(rows, by=groups_box.value))}</pre>"


def show():
    for ring in state["rings"]:
        ring.remove()
    if state["highlight"] not in {m for *_, m in state["hits"]}:
        state["highlight"] = None
    state["rings"] = V.highlight(state["hits"], state["highlight"])
    picture = io.BytesIO()
    state["fig"].savefig(picture, format="png", dpi=100, facecolor="white")
    image.value = picture.getvalue()
    picked.value = f"highlighted: {state['highlight']}" if state["highlight"] else ""


def save(_):
    if state["fig"] is None:
        status.value = "nothing to save"
        return
    target = Path(path_field.value).expanduser()
    target.parent.mkdir(parents=True, exist_ok=True)
    state["fig"].savefig(target, dpi=200, facecolor="white", bbox_inches="tight")
    status.value = f"saved {target}"


def on_click(event):
    x, y = event.get("dataX"), event.get("dataY")
    if x is None or not state["hits"]:
        return
    scale = 100 / state["fig"].dpi
    distance, model = min((np.hypot(hx * scale - x, hy * scale - y), m) for _, hx, hy, m in state["hits"])
    state["highlight"] = model if distance < 12 and model != state["highlight"] else None
    show()


try:
    from ipyevents import Event
    CLICKS = Event(source=image, watched_events=["click"])
    CLICKS.on_dom_event(on_click)
except ImportError:
    CLICKS = None

split_box.observe(set_split, names="value")
poses_box.observe(set_choices, names="value")
for box in (occlusion_box, style_box, model_fronts_box, labels_box, groups_box):
    box.observe(redraw, names="value")
save_button.on_click(save)
display(widgets.VBox([widgets.HBox([split_box, poses_box, occlusion_box]), widgets.HBox([groups_box, style_box]), widgets.HBox([model_fronts_box, labels_box]),
                      model_boxes, widgets.HBox([path_field, save_button, status]), picked, image, text]))
set_choices()

## 4. Reading

To be written with the results of the studies on the 1000 validation poses (`rsgm_inpaint_pareto`,
`rsgm_inpaint_mean_apd_*`): copy their `.db` files from the servers into `optuna/` under `RESULTS`.